In [1]:
from splinter import Browser
from bs4 import BeautifulSoup as soup 
import re
import pandas as pd
import numpy as np
import time
import json
import random

In [10]:
browser = Browser('chrome')
city = "Chandigarh"
target_cars = 600
cars_collected = 0
total_pages = 250 

In [11]:
def collect_car_links(city, total_pages, target_cars):

    cars_collected = 0

    with open(f"car_links_{city}.txt", "a") as f:

        for page_num in range(1, total_pages + 1):

            if page_num == 1:
                url = f"https://www.cardekho.com/used-cars+in+{city}"
            else:
                url = f"https://www.cardekho.com/used-cars+in+{city}/page-{page_num}"

            browser.visit(url)
            time.sleep(2)

            browser.execute_script("window.scrollTo(0, 1000);")
            time.sleep(1)

            current_soup = soup(browser.html, 'html.parser')

            page_links_found = 0

            for link in current_soup.find_all('a', href=True):
                href = link['href']

                if 'used-car-details' in href:

                    full_url = f"https://www.cardekho.com{href}" if href.startswith('/') else href

                    f.write(full_url + "\n")

                    cars_collected += 1
                    page_links_found += 1

            print(f"Page {page_num}: Saved {page_links_found} links. Total: {cars_collected}")

            if cars_collected >= target_cars:
                print("Target reached!")
                break

    print(f"All links are now saved in car_links_{city.lower()}.txt")

In [12]:
collect_car_links(city, total_pages, target_cars)

Page 1: Saved 28 links. Total: 28
Page 2: Saved 28 links. Total: 56
Page 3: Saved 28 links. Total: 84
Page 4: Saved 28 links. Total: 112
Page 5: Saved 28 links. Total: 140
Page 6: Saved 28 links. Total: 168
Page 7: Saved 28 links. Total: 196
Page 8: Saved 28 links. Total: 224
Page 9: Saved 28 links. Total: 252
Page 10: Saved 28 links. Total: 280
Page 11: Saved 28 links. Total: 308
Page 12: Saved 28 links. Total: 336
Page 13: Saved 28 links. Total: 364
Page 14: Saved 28 links. Total: 392
Page 15: Saved 28 links. Total: 420
Page 16: Saved 28 links. Total: 448
Page 17: Saved 28 links. Total: 476
Page 18: Saved 28 links. Total: 504
Page 19: Saved 28 links. Total: 532
Page 20: Saved 28 links. Total: 560
Page 21: Saved 28 links. Total: 588
Page 22: Saved 28 links. Total: 616
Target reached!
All links are now saved in car_links_chandigarh.txt


## Main Extraction 

In [13]:
def scrape_car_links(city, links_filename="car_links.txt"):

    def get_browser():
        return Browser('chrome')

    # 1. Load links
    with open(links_filename, "r") as f:
        all_links = [line.strip() for line in f.readlines()]

    output_file = f"car_dataset_{city.lower()}.json"
    browser = get_browser()

    for index, link in enumerate(all_links):
        try:
            print(f"Scraping {index+1}/{len(all_links)}: {link}")

            # Visit page
            browser.visit(link)

            # Human delay + scroll
            time.sleep(random.uniform(1, 2))
            browser.execute_script("window.scrollTo(0, 600);")
            time.sleep(0.5)

            # Expand specifications
            try:
                view_all_spec_btn = browser.find_by_text('View all Specifications')
                if view_all_spec_btn:
                    browser.execute_script(
                        "arguments[0].click();",
                        view_all_spec_btn.first._element
                    )
                    print("Expanded specifications.")
                    time.sleep(0.6)
            except Exception:
                pass

            # Parse page
            page_soup = soup(browser.html, 'html.parser')

            car_data = {"url": link}

            # -------------------------
            # CAR NAME EXTRACTION
            # -------------------------
            name_tag = page_soup.find('div', class_='vehicleName')
            h1 = name_tag.find('h1') if (name_tag and name_tag.find('h1')) else page_soup.find('h1')

            if h1:
                parts = h1.get_text(separator="|", strip=True).split("|")
                car_data["car_name"] = parts[1].strip() if len(parts) >= 2 else parts[0].strip()

            # -------------------------
            # PRICE EXTRACTION
            # -------------------------
            price_div = page_soup.find('div', class_='vehiclePrice')
            if price_div:
                price_span = price_div.find('span')
                if price_span:
                    car_data["Price"] = price_span.get_text(strip=True)

            # -------------------------
            # SPECIFICATIONS EXTRACTION
            # -------------------------
            spec_items = page_soup.find_all('li', class_='gsc_col-xs-12')

            for item in spec_items:
                label_tag = item.find('div', class_='label')
                value_tag = item.find('span', class_='value-text')

                if label_tag and value_tag:
                    label = label_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    car_data[label] = value

            # Save data
            if len(car_data) > 1:
                with open(output_file, "a") as out:
                    out.write(json.dumps(car_data) + "\n")

                print(f"Saved: {car_data.get('Price','N/A')} and {len(car_data)-2} specs.")
            else:
                print(f"No data found for: {link}")

        except Exception as e:
            print(f"Serious error at {link}: {e}")

            browser.quit()
            browser = get_browser()
            time.sleep(1)
            continue

    browser.quit()

In [14]:
scrape_car_links(f"{city}", f"car_links_{city}.txt")

Scraping 1/616: https://www.cardekho.com/used-car-details/used-Hyundai-creta-sx-ivt-bsvi-cars-Chandigarh_e8eee7c0-cdfe-46f8-a58d-d71fd469562a.htm?adId=23410&adType=41
Expanded specifications.
Saved: ₹11 Lakh and 43 specs.
Scraping 2/616: https://www.cardekho.com/buy-used-car-details/used-Hyundai-alcazar-platinum-ae-turbo-7str-cars-Chandigarh_b1edb6fa-3b9a-447c-8ed1-571fb68f7ee2.htm
Expanded specifications.
Saved: ₹14.19 Lakh and 46 specs.
Scraping 3/616: https://www.cardekho.com/buy-used-car-details/used-Kia-seltos-gtx-plus-s-turbo-dct-cars-Chandigarh_edc737e6-cd31-43d2-beee-68b766f8745f.htm
Expanded specifications.
Saved: ₹10.09 Lakh and 44 specs.
Scraping 4/616: https://www.cardekho.com/buy-used-car-details/used-Honda-wr-v-i-vtec-vx-cars-Chandigarh_08f8c01d-093a-45df-850d-73e86ec3f610.htm
Expanded specifications.
Saved: ₹5.67 Lakh and 54 specs.
Scraping 5/616: https://www.cardekho.com/used-car-details/used-Renault-kwid-10-rxl-opt-cars-Chandigarh_61f515a5-7028-4f65-9ca0-039c2caf2dae.h